# Summarization and Visualization

This notebook mirrors the summarize stage of the pipeline, but keeps the workflow display-only. It loads the pipeline accuracy dataframe, splits the publication-era models from the newly added models, displays separate grouped summary tables, and shows separate Sankey diagrams for the old and new model groups.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import HTML, display

import plotly.graph_objects as go

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "ez-pipeline" / "pipeline" / "pipeline_summary.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing ez-pipeline/pipeline/pipeline_summary.py")

REPO_ROOT = find_repo_root(Path.cwd())
PIPELINE_DIR = REPO_ROOT / "ez-pipeline" / "pipeline"

if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

import pipeline_config as cfg
from pipeline_accuracy import compute_accuracy_metrics, get_accuracy_df
from pipeline_summary import COL_SPEC, make_sankey_figure

pd.set_option("display.max_rows", None)

OLD_MODELS = [
    "gpt-4.1",
    "gpt-4o",
    "gpt-5",
    "gpt-o3",
    "claude-4-opus-20250514",
]

NEW_MODELS = [
    "gpt-5.4",
    "claude-opus-4-7",
    "gpt-5.5",
]

ALL_MODELS = OLD_MODELS + NEW_MODELS
PROMPT_ORDER = ["prompt1", "prompt2", "prompt3"]

def filter_models(df: pd.DataFrame, models: list[str]) -> pd.DataFrame:
    return df[df["model"].isin(models)].copy()

def build_summary_table_for_models(master_df: pd.DataFrame, models: list[str]) -> pd.DataFrame:
    grand_totals = {key: 0 for key in COL_SPEC}
    model_totals = {model: {key: 0 for key in COL_SPEC} for model in models}
    rows: list[dict[str, object]] = []

    for model_name in models:
        model_label = cfg.MODEL_DISPLAY.get(model_name, model_name)
        for prompt_name in PROMPT_ORDER:
            combo_df = get_accuracy_df(prompt_name, model_name, master_df)
            if combo_df.empty:
                continue

            metrics = compute_accuracy_metrics(combo_df)
            row_values = {column: int(spec(metrics)) for column, spec in COL_SPEC.items()}
            for column, value in row_values.items():
                grand_totals[column] += value
                model_totals[model_name][column] += value
            rows.append({"Model": model_label, "Prompt": prompt_name.replace("prompt", "P"), **row_values})

    out_rows = [{"Model": "", "Prompt": f"FULL {len(master_df)}", **grand_totals}]
    detail_df = pd.DataFrame(rows)
    if not detail_df.empty:
        detail_df = detail_df.set_index(["Model", "Prompt"])

    for model_name in models:
        model_label = cfg.MODEL_DISPLAY.get(model_name, model_name)
        out_rows.append({"Model": model_label, "Prompt": "TOTAL", **model_totals[model_name]})
        for prompt_name in PROMPT_ORDER:
            prompt_label = prompt_name.replace("prompt", "P")
            if not detail_df.empty and (model_label, prompt_label) in detail_df.index:
                out_rows.append({"Model": model_label, "Prompt": prompt_label, **detail_df.loc[(model_label, prompt_label)].to_dict()})

    summary_df = pd.DataFrame(out_rows).set_index(["Model", "Prompt"])
    return summary_df[list(COL_SPEC)]

PUBLICATION_TABLE_ROWS = [
    "Sanitizer",
    "Parser",
    "Execution/PS",
    "Accuracy/PS",
]

def _publication_row_values(df: pd.DataFrame) -> dict[str, object]:
    metrics = compute_accuracy_metrics(df)
    sanitizer_pass = metrics.get("sanitizer_to_parser", 0)
    parser_pass = metrics.get("parser_to_execution", 0)
    direct_execution = metrics.get("execution_to_accuracy", 0)
    psz_execution = metrics.get("execution_to_accuracy_after_pairstyle", 0)
    direct_accurate = metrics.get("accuracy_to_correct", 0)
    psz_accurate = metrics.get("pair_accuracy_to_correct", 0)

    return {
        "Sanitizer": int(sanitizer_pass),
        "Parser": int(parser_pass),
        "Execution/PS": f"{int(direct_execution)}/{int(psz_execution)}",
        "Accuracy/PS": f"{int(direct_accurate)}/{int(psz_accurate)}",
    }

def build_publication_table_for_models(
    master_df: pd.DataFrame,
    models: list[str],
    include_prompt_breakdown: bool = False,
) -> pd.DataFrame:
    rows: list[dict[str, object]] = []

    for model_name in models:
        model_df = filter_models(master_df, [model_name])
        model_label = cfg.MODEL_DISPLAY.get(model_name, model_name)
        if model_df.empty:
            continue

        if include_prompt_breakdown:
            rows.append({"Model": model_label, "Prompt": "TOTAL", **_publication_row_values(model_df)})
            for prompt_name in PROMPT_ORDER:
                prompt_df = model_df[model_df["prompt"] == prompt_name]
                if prompt_df.empty:
                    continue
                rows.append(
                    {
                        "Model": model_label,
                        "Prompt": prompt_name.replace("prompt", "P"),
                        **_publication_row_values(prompt_df),
                    }
                )
        else:
            rows.append({"Model": model_label, **_publication_row_values(model_df)})

    if include_prompt_breakdown:
        table_df = pd.DataFrame(rows).set_index(["Model", "Prompt"])
    else:
        table_df = pd.DataFrame(rows).set_index("Model")
    return table_df[PUBLICATION_TABLE_ROWS]

SANKEY_NODE_LABELS = [
    "Normalization",
    "Parser",
    "Execution",
    "Accuracy",
    "",
    "PSZ",
    "Acc.",
    "",
    "",
    "",
    "",
    "",
    "",
]


def make_new_model_sankey_figure(metrics: dict[str, int]) -> go.Figure:
    """Editable Sankey layout for the new-model subset.

    This copy omits the PSZ accuracy-failure terminal because the new-model
    subset currently has zero scripts in that category. Adjust `x` and `y`
    below to fine-tune node placement for the publication figure.
    """
    return go.Figure(
        go.Sankey(
            arrangement="fixed",
            node=dict(
                pad=22,
                thickness=36,
                label=[
                    "Normalization",
                    "Parser",
                    "Execution",
                    "Accuracy",
                    "",
                    "PSZ",
                    "",
                    "",
                    "",
                    "",
                    "",
                    "",
                ],
                x=[0.02, 0.22, 0.42, 0.70, 0.93, 0.58, 0.78, 0.10, 0.32, 0.72, 0.88, 0.93],
                y=[0.46, 0.42, 0.42, 0.16, 0.08, 0.68, 0.60, 0.92, 0.92, 0.88, 0.32, 0.56],
                color=["black"] * 12,
            ),
            link=dict(
                source=[0, 0, 1, 1, 2, 2, 5, 5, 3, 3, 6],
                target=[1, 7, 2, 8, 3, 5, 6, 9, 4, 10, 11],
                value=[
                    metrics["sanitizer_to_parser"],
                    metrics["sanitizer_to_failure"],
                    metrics["parser_to_execution"],
                    metrics["parser_to_failure"],
                    metrics["execution_to_accuracy"],
                    metrics["execution_to_pairstylecheck"],
                    metrics["execution_to_accuracy_after_pairstyle"],
                    metrics["execution_to_pairstylecheck"] - metrics["execution_to_accuracy_after_pairstyle"],
                    metrics["accuracy_to_correct"],
                    metrics["accuracy_to_failure"],
                    metrics["pair_accuracy_to_correct"],
                ],
                color=[
                    "rgba(34, 139, 34, 0.4)",
                    "rgba(178, 34, 34, 0.4)",
                    "rgba(34, 139, 34, 0.4)",
                    "rgba(178, 34, 34, 0.4)",
                    "rgba(34, 139, 34, 0.4)",
                    "rgba(218, 165, 32, 0.4)",
                    "rgba(34, 139, 34, 0.4)",
                    "rgba(178, 34, 34, 0.4)",
                    "rgba(34, 139, 34, 0.4)",
                    "rgba(178, 34, 34, 0.4)",
                    "rgba(126, 152, 33, 0.4)",
                ],
            ),
        )
    ).update_layout(
        title="LAMMPS Evaluation Pipeline Flow: New 3 Models",
        font=dict(size=14),
        width=1200,
        height=700,
    )

def display_new_model_sankey(df: pd.DataFrame) -> go.Figure:
    model_df = filter_models(df, NEW_MODELS)
    metrics = compute_accuracy_metrics(model_df)
    fig = make_new_model_sankey_figure(metrics)
    display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
    return fig

def display_sankey_for_models(df: pd.DataFrame, models: list[str], title: str) -> go.Figure:
    model_df = filter_models(df, models)
    metrics = compute_accuracy_metrics(model_df)
    fig = make_sankey_figure(metrics)
    fig.data[0].node.label = SANKEY_NODE_LABELS
    fig.update_layout(title=title)
    display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
    return fig

print(f"Repo root: {REPO_ROOT}")
print(f"Accuracy dataframe: {cfg.ACCURACY_DF_PATH}")


Repo root: /scratch/gilbreth/holbrooe/LAMMPS-AST
Accuracy dataframe: /scratch/gilbreth/holbrooe/LAMMPS-AST/publication/pipeline_generated_files/data/accuracy_df_trees.pkl


In [2]:
accuracy_df = pd.read_pickle(cfg.ACCURACY_DF_PATH)

old_accuracy_df = filter_models(accuracy_df, OLD_MODELS)
new_accuracy_df = filter_models(accuracy_df, NEW_MODELS)
all_accuracy_df = filter_models(accuracy_df, ALL_MODELS)

print(f"Old-model rows: {len(old_accuracy_df)}")
print(f"New-model rows: {len(new_accuracy_df)}")
print(f"All-model rows: {len(all_accuracy_df)}")

display(accuracy_df)


Old-model rows: 150
New-model rows: 90
All-model rows: 240


,prompt,model,trial,sanitized,parsed,ast_path,run,pair_run,accurate
0,prompt1,gpt-4.1,0,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,['ERROR: Incorrect args for pair coefficients ...,True,"PSZ, pair_style inaccurate"
1,prompt1,gpt-4.1,1,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,True,n/a,True
2,prompt1,gpt-4.1,2,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,True,n/a,True
3,prompt1,gpt-4.1,3,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,True,n/a,True
4,prompt1,gpt-4.1,4,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,True,n/a,True
5,prompt1,gpt-4.1,5,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,True,n/a,True
6,prompt1,gpt-4.1,6,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,True,n/a,True
7,prompt1,gpt-4.1,7,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,True,n/a,True
8,prompt1,gpt-4.1,8,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,['ERROR: Incorrect args for pair coefficients ...,True,"PSZ, pair_style inaccurate"
9,prompt1,gpt-4.1,9,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,['ERROR: Incorrect args for pair coefficients ...,True,"PSZ, pair_style inaccurate"


In [3]:
# new_accuracy_df[new_accuracy_df['prompt']=='prompt3']
new_accuracy_df

,prompt,model,trial,sanitized,parsed,ast_path,run,pair_run,accurate
50,prompt1,gpt-5.4,0,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,['ERROR: Run_style command before simulation b...,False,unknown error
51,prompt1,gpt-5.4,1,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,True,n/a,True
52,prompt1,gpt-5.4,2,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,['ERROR: Run_style command before simulation b...,False,unknown error
53,prompt1,gpt-5.4,3,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,True,n/a,True
54,prompt1,gpt-5.4,4,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,True,n/a,True
55,prompt1,gpt-5.4,5,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,['ERROR: Run_style command before simulation b...,False,unknown error
56,prompt1,gpt-5.4,6,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,True,n/a,True
57,prompt1,gpt-5.4,7,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,['ERROR: Run_style command before simulation b...,False,unknown error
58,prompt1,gpt-5.4,8,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,['ERROR: Run_style command before simulation b...,False,unknown error
59,prompt1,gpt-5.4,9,True,True,/scratch/gilbreth/holbrooe/LAMMPS-AST/publicat...,True,n/a,True


In [4]:
old_summary_df = build_summary_table_for_models(old_accuracy_df, OLD_MODELS)
display(old_summary_df)
old_summary_df.to_csv("summary_table_old_models.csv", index=True)


Correct_Acc  Failure_Acc  Correct_PSZ  \
Model                  Prompt                                            
                       FULL 150           41            7           18   
gpt-4.1                TOTAL               7            1            4   
                       P1                  7            0            3   
                       P2                  0            1            1   
                       P3                  0            0            0   
gpt-4o                 TOTAL               7            1            3   
                       P1                  7            0            3   
                       P2                  0            1            0   
                       P3                  0            0            0   
gpt-5                  TOTAL              10            2            3   
                       P1                  6            2            0   
                       P2                  3            0            3   
                       P3                  1            0            0   
gpt-o3                 TOTAL               8            2            0   
                       P1                  4            0            0   
                       P2                  4            1            0   
                       P3                  0            1            0   
claude-4-opus-20250514 TOTAL               9            1            8   
                       P1                  9            1            0   
                       P2                  0            0            8   
                       P3                  0            0            0   

                                 Failure_PSZ  Failure_PSZ_Exec  \
Model                  Prompt                                    
                       FULL 150           16                29   
gpt-4.1                TOTAL               9                 3   
                       P1                  0                 0   
                       P2                  7                 0   
                       P3                  2                 3   
gpt-4o                 TOTAL               5                 7   
                       P1                  0                 0   
                       P2                  5                 1   
                       P3                  0                 6   
gpt-5                  TOTAL               0                 3   
                       P1                  0                 2   
                       P2                  0                 1   
                       P3                  0                 0   
gpt-o3                 TOTAL               0                 7   
                       P1                  0                 2   
                       P2                  0                 3   
                       P3                  0                 2   
claude-4-opus-20250514 TOTAL               2                 9   
                       P1                  0                 0   
                       P2                  2                 0   
                       P3                  0                 9   

                                 Failure_parser  Failure_sanitizer  
Model                  Prompt                                       
                       FULL 150              35                  4  
gpt-4.1                TOTAL                  6                  0  
                       P1                     0                  0  
                       P2                     1                  0  
                       P3                     5                  0  
gpt-4o                 TOTAL                  7                  0  
                       P1                     0                  0  
                       P2                     3                  0  
                       P3                     4                  0  
gpt-5                  TOTAL                  9     

In [5]:
new_summary_df = build_summary_table_for_models(new_accuracy_df, NEW_MODELS)
display(new_summary_df)
new_summary_df.to_csv("summary_table_new_models.csv", index=True)


Correct_Acc  Failure_Acc  Correct_PSZ  Failure_PSZ  \
Model           Prompt                                                        
                FULL 90           41            9            5            0   
gpt-5.4         TOTAL              7            1            0            0   
                P1                 5            0            0            0   
                P2                 2            0            0            0   
                P3                 0            1            0            0   
claude-opus-4-7 TOTAL             20            4            0            0   
                P1                10            0            0            0   
                P2                10            0            0            0   
                P3                 0            4            0            0   
gpt-5.5         TOTAL             14            4            5            0   
                P1                10            0            0            0   
                P2                 4            0            5            0   
                P3                 0            4            0            0   

                         Failure_PSZ_Exec  Failure_parser  Failure_sanitizer  
Model           Prompt                                                        
                FULL 90                27               5                  3  
gpt-5.4         TOTAL                  16               4                  2  
                P1                      5               0                  0  
                P2                      7               1                  0  
                P3                      4               3                  2  
claude-opus-4-7 TOTAL                   6               0                  0  
                P1                      0               0                  0  
                P2                      0               0                  0  
                P3                      6               0                  0  
gpt-5.5         TOTAL                   5               1                  1  
                P1                      0               0                  0  
                P2                      0               0                  1  
                P3                      5               1                  0

In [73]:
all_summary_df = build_summary_table_for_models(all_accuracy_df, ALL_MODELS)
display(all_summary_df)
all_summary_df.to_csv("output.csv", index=True)


Correct_Acc  Failure_Acc  Correct_PSZ  \
Model                  Prompt                                            
                       FULL 240           82           16           23   
gpt-4.1                TOTAL               7            1            4   
                       P1                  7            0            3   
                       P2                  0            1            1   
                       P3                  0            0            0   
gpt-4o                 TOTAL               7            1            3   
                       P1                  7            0            3   
                       P2                  0            1            0   
                       P3                  0            0            0   
gpt-5                  TOTAL              10            2            3   
                       P1                  6            2            0   
                       P2                  3            0            3   
                       P3                  1            0            0   
gpt-o3                 TOTAL               8            2            0   
                       P1                  4            0            0   
                       P2                  4            1            0   
                       P3                  0            1            0   
claude-4-opus-20250514 TOTAL               9            1            8   
                       P1                  9            1            0   
                       P2                  0            0            8   
                       P3                  0            0            0   
gpt-5.4                TOTAL               7            1            0   
                       P1                  5            0            0   
                       P2                  2            0            0   
                       P3                  0            1            0   
claude-opus-4-7        TOTAL              20            4            0   
                       P1                 10            0            0   
                       P2                 10            0            0   
                       P3                  0            4            0   
gpt-5.5                TOTAL              14            4            5   
                       P1                 10            0            0   
                       P2                  4            0            5   
                       P3                  0            4            0   

                                 Failure_PSZ  Failure_PSZ_Exec  \
Model                  Prompt                                    
                       FULL 240           16                56   
gpt-4.1                TOTAL               9                 3   
                       P1                  0                 0   
                       P2                  7                 0   
                       P3                  2                 3   
gpt-4o                 TOTAL               5                 7   
                       P1                  0                 0   
                       P2                  5                 1   
                       P3                  0                 6   
gpt-5                  TOTAL               0                 3   
                       P1                  0                 2   
                       P2                  0                 1   
                       P3                  0                 0   
gpt-o3                 TOTAL               0                 7   
                       P1                  0                 2   
                       P2                  0                 3   
                       P3                  0                 2   
claude-4-opus-20250514 TOTAL               2                 9   
                       P1                  0                 0   
                       P2                  2            

In [74]:
old_publication_table_df = build_publication_table_for_models(old_accuracy_df, OLD_MODELS)
display(old_publication_table_df)
old_publication_table_df.to_csv("publication_table_old_models.csv", index=True)


,Sanitizer,Parser,Execution/PS,Accuracy/PS
Model,,,,
gpt-4.1,30,24,8/13,7/4
gpt-4o,30,23,8/8,7/3
gpt-5,27,18,12/3,10/3
gpt-o3,29,17,10/0,8/0
claude-4-opus-20250514,30,29,10/10,9/8


In [75]:
new_publication_table_df = build_publication_table_for_models(new_accuracy_df, NEW_MODELS)
display(new_publication_table_df)
new_publication_table_df.to_csv("publication_table_new_models.csv", index=True)


,Sanitizer,Parser,Execution/PS,Accuracy/PS
Model,,,,
gpt-5.4,28,24,8/0,7/0
claude-opus-4-7,30,30,24/0,20/0
gpt-5.5,29,28,18/5,14/5


In [76]:
all_publication_table_df = build_publication_table_for_models(
    all_accuracy_df,
    ALL_MODELS,
    include_prompt_breakdown=True,
)
display(all_publication_table_df)
all_publication_table_df.to_csv("publication_table_all_models.csv", index=True)


Sanitizer  Parser Execution/PS Accuracy/PS
Model                  Prompt                                            
gpt-4.1                TOTAL          30      24         8/13         7/4
                       P1             10      10          7/3         7/3
                       P2             10       9          1/8         0/1
                       P3             10       5          0/2         0/0
gpt-4o                 TOTAL          30      23          8/8         7/3
                       P1             10      10          7/3         7/3
                       P2             10       7          1/5         0/0
                       P3             10       6          0/0         0/0
gpt-5                  TOTAL          27      18         12/3        10/3
                       P1             10      10          8/0         6/0
                       P2              7       7          3/3         3/3
                       P3             10       1          1/0         1/0
gpt-o3                 TOTAL          29      17         10/0         8/0
                       P1             10       6          4/0         4/0
                       P2              9       8          5/0         4/0
                       P3             10       3          1/0         0/0
claude-4-opus-20250514 TOTAL          30      29        10/10         9/8
                       P1             10      10         10/0         9/0
                       P2             10      10         0/10         0/8
                       P3             10       9          0/0         0/0
gpt-5.4                TOTAL          28      24          8/0         7/0
                       P1             10      10          5/0         5/0
                       P2             10       9          2/0         2/0
                       P3              8       5          1/0         0/0
claude-opus-4-7        TOTAL          30      30         24/0        20/0
                       P1             10      10         10/0        10/0
                       P2             10      10         10/0        10/0
                       P3             10      10          4/0         0/0
gpt-5.5                TOTAL          29      28         18/5        14/5
                       P1             10      10         10/0        10/0
                       P2              9       9          4/5         4/5
                       P3             10       9          4/0         0/0

In [77]:
latex_table = all_publication_table_df.to_latex(
    index=True,
    escape=False,
    multicolumn=True,
    multirow=True,
    caption=(
        "LAMMPS evaluation pipeline outcomes by model and prompt. "
        "Execution/PS reports direct execution successes followed by successes after pair-style-zero substitution; "
        "Accuracy/PS reports direct accuracy successes followed by accuracy successes after pair-style-zero substitution."
    ),
    label="tab:lammps_pipeline_model_prompt",
)

print(latex_table)
Path("publication_table_all_models.tex").write_text(latex_table, encoding="utf-8")


\begin{table}
\caption{LAMMPS evaluation pipeline outcomes by model and prompt. Execution/PS reports direct execution successes followed by successes after pair-style-zero substitution; Accuracy/PS reports direct accuracy successes followed by accuracy successes after pair-style-zero substitution.}
\label{tab:lammps_pipeline_model_prompt}
\begin{tabular}{llrrll}
\toprule
 &  & Sanitizer & Parser & Execution/PS & Accuracy/PS \\
Model & Prompt &  &  &  &  \\
\midrule
\multirow[t]{4}{*}{gpt-4.1} & TOTAL & 30 & 24 & 8/13 & 7/4 \\
 & P1 & 10 & 10 & 7/3 & 7/3 \\
 & P2 & 10 & 9 & 1/8 & 0/1 \\
 & P3 & 10 & 5 & 0/2 & 0/0 \\
\cline{1-6}
\multirow[t]{4}{*}{gpt-4o} & TOTAL & 30 & 23 & 8/8 & 7/3 \\
 & P1 & 10 & 10 & 7/3 & 7/3 \\
 & P2 & 10 & 7 & 1/5 & 0/0 \\
 & P3 & 10 & 6 & 0/0 & 0/0 \\
\cline{1-6}
\multirow[t]{4}{*}{gpt-5} & TOTAL & 27 & 18 & 12/3 & 10/3 \\
 & P1 & 10 & 10 & 8/0 & 6/0 \\
 & P2 & 7 & 7 & 3/3 & 3/3 \\
 & P3 & 10 & 1 & 1/0 & 1/0 \\
\cline{1-6}
\multirow[t]{4}{*}{gpt-o3} & TOTAL & 29

1855

In [78]:
old_sankey_fig = display_sankey_for_models(
    accuracy_df,
    OLD_MODELS,
    "LAMMPS Evaluation Pipeline Flow: Original 5 Models",
)


In [79]:
new_sankey_fig = display_new_model_sankey(accuracy_df)


In [80]:
from IPython.display import HTML, display

display(HTML("""
<script>
(function () {
  const roots = Array.from(document.querySelectorAll('.js-plotly-plot'));
  if (!roots.length) return;
  roots.forEach((root) => {
  // ---- Font sizing controls ----
  const FS_MIN = 14;   // px (floor)
  const FS_MAX = 30;   // px (cap)
  const FS_K   = 0.12; // scale factor (px font per px bar height)
  // NEW: inset (px) for placing numbers just outside the edge
  const INSET = 6;

  function parseBase(transformStr) {
    const m = /translate\\(\\s*([-+]?\\d*\\.?\\d+)\\s*,\\s*([-+]?\\d*\\.?\\d+)\\s*\\)/.exec(transformStr || "");
    return { x: m ? parseFloat(m[1]) || 0 : 0, y: m ? parseFloat(m[2]) || 0 : 0 };
  }

  // number formatter
  const fmt = new Intl.NumberFormat(undefined, { maximumFractionDigits: 0 });

  function placeLabels() {
    const nodes = root.querySelectorAll('g.sankey g.sankey-node-set g.sankey-node');
    // compute node totals (max(in, out)) from the plotted trace
    let tot = [];
    const tr = (root._fullData || []).find(t => t.type === 'sankey');
    if (tr) {
      const n = tr.node.label.length;
      const inTot  = Array(n).fill(0);
      const outTot = Array(n).fill(0);
      const S = tr.link.source || [];
      const T = tr.link.target || [];
      const V = tr.link.value  || [];
      for (let i = 0; i < V.length; i++) {
        outTot[S[i]] += +V[i] || 0;
        inTot[T[i]]  += +V[i] || 0;
      }
      tot = inTot.map((v,i) => Math.max(v, outTot[i]));
    }

    nodes.forEach((node, i) => {
      const rect  = node.querySelector('rect.node-rect');
      const label = node.querySelector('text.node-label');
      if (!rect || !label) return;

      const w = parseFloat(rect.getAttribute('width'))  || 0;
      const h = parseFloat(rect.getAttribute('height')) || 0;

      // --- Scaled & capped font size ---
      let fontSize = Math.round(h * FS_K);
      if (!isFinite(fontSize)) fontSize = FS_MIN;
      fontSize = Math.max(FS_MIN, Math.min(FS_MAX, fontSize));

      // Preserve original side/transform from first run
      const origAnchor    = label.dataset.origAnchor    || label.getAttribute('text-anchor') || 'start';
      const origTransform = label.dataset.origTransform || (label.getAttribute('transform') || '');
      label.dataset.origAnchor = origAnchor;
      label.dataset.origTransform = origTransform;

      // Style the rotated label (white) – stays centered inside bar
      label.style.fill = '#fff';
      label.style.fontWeight = '700';
      label.style.fontSize = fontSize + 'px';
      label.style.textShadow = 'none';
      label.style.writingMode = 'sideways-lr';   // vertical, preserves Plotly's translate(x, midY)
      label.style.textOrientation = 'mixed';
      label.style.textRendering = 'geometricPrecision';
      label.style.webkitFontSmoothing = 'antialiased';
      label.style.MozOsxFontSmoothing = 'grayscale';
      label.setAttribute('text-anchor', 'middle');
      label.setAttribute('dominant-baseline', 'middle');

      // Keep Plotly's mid-Y; shift X to bar center — based on the ORIGINAL transform
      const {x: x0} = parseBase(origTransform);
      const xTarget = w / 2;
      const dxCenter = Math.round(xTarget - x0);  // snap to integer px to avoid blur
      label.setAttribute('transform', origTransform + ` translate(${dxCenter},0)`);

      // --- Number on the LEFT side of the bar (black, not rotated) ---
      let vtext = node.querySelector('text.__nodeval');
      if (!vtext) {
        vtext = document.createElementNS('http://www.w3.org/2000/svg','text');
        vtext.setAttribute('class','__nodeval');
        node.appendChild(vtext);
      }
      const valSize = Math.max(12, Math.min(22, fontSize));

      // ---- MODIFIED SECTION ----
      // Always place the number on the LEFT of the bar.
      // To do this, we right-align the text ('end' anchor) and place it just to the left of the bar's origin.
      vtext.setAttribute('text-anchor', 'end'); // Use 'end' to right-align the text
      vtext.setAttribute('dominant-baseline', 'middle');
      vtext.setAttribute('style',
        `font-size:${valSize}px;font-weight:700;fill:#000;text-shadow:none;pointer-events:none;writing-mode:initial;text-orientation:mixed;`);

      // Compute target x on the left side relative to the node's local coords
      const desiredX = -INSET; // Position is always -INSET (just left of the bar)
      const dxOther  = Math.round(desiredX - x0);
      
      // Position number at same mid-Y as original, but on the left side in X
      vtext.setAttribute('transform', origTransform + ` translate(${dxOther},0)`);

      // Content
      const val = (tot && Number.isFinite(tot[i])) ? tot[i] : 0;
      vtext.textContent = fmt.format(val);
    });
  }

  if (root.on) {
    root.on('plotly_afterplot', placeLabels);
    root.on('plotly_relayout',  placeLabels);
    root.on('plotly_animated',  placeLabels);
  }
  placeLabels();
  });
})();
</script>
"""))